In [6]:
print("Hello World")

Hello World


In [7]:
%pip install -U torch torchvision transformers pillow tqdm scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoProcessor, CLIPModel

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [9]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Using device: cuda


In [10]:
DATASET_ROOT = r"C:\AI\Medical\Liver\data\lungs-disease-dataset-4-types\versions\1\Lung Disease Dataset"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)
print("Test:", TEST_DIR)

Train: C:\AI\Medical\Liver\data\lungs-disease-dataset-4-types\versions\1\Lung Disease Dataset\train
Val: C:\AI\Medical\Liver\data\lungs-disease-dataset-4-types\versions\1\Lung Disease Dataset\val
Test: C:\AI\Medical\Liver\data\lungs-disease-dataset-4-types\versions\1\Lung Disease Dataset\test


In [11]:
CLASSES = [
    "Bacterial Pneumonia",
    "Corona Virus Disease",
    "Normal",
    "Tuberculosis",
    "Viral Pneumonia"
]

CLASS_TO_IDX = {
    class_name: i
    for i, class_name in enumerate(CLASSES)
}

IDX_TO_CLASS = {
    i: class_name
    for i, class_name in enumerate(CLASSES)
}

print(CLASS_TO_IDX)

{'Bacterial Pneumonia': 0, 'Corona Virus Disease': 1, 'Normal': 2, 'Tuberculosis': 3, 'Viral Pneumonia': 4}


In [12]:
for split_name, split_dir in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR)
]:
    
    print("\n", "=" * 50)
    print(split_name)
    print("=" * 50)
    
    for class_name in CLASSES:
        
        class_dir = os.path.join(split_dir, class_name)
        
        if os.path.exists(class_dir):
            files = [
                f for f in os.listdir(class_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
            ]
            
            print(f"{class_name}: {len(files)}")
            
        else:
            print(f"{class_name}: FOLDER NOT FOUND")


TRAIN
Bacterial Pneumonia: 1205
Corona Virus Disease: 1218
Normal: 1207
Tuberculosis: 1220
Viral Pneumonia: 1204

VAL
Bacterial Pneumonia: 401
Corona Virus Disease: 406
Normal: 402
Tuberculosis: 406
Viral Pneumonia: 401

TEST
Bacterial Pneumonia: 403
Corona Virus Disease: 407
Normal: 404
Tuberculosis: 408
Viral Pneumonia: 403


In [13]:
MODEL_NAME = "openai/clip-vit-base-patch32"

print("Loading CLIP processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

print("Loading CLIP model...")

clip_model = CLIPModel.from_pretrained(
    MODEL_NAME
)

clip_model = clip_model.to(DEVICE)

print("CLIP loaded.")

Loading CLIP processor...
Loading CLIP model...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 29221.44it/s]


CLIP loaded.


In [14]:
clip_model.eval()

for param in clip_model.parameters():
    param.requires_grad = False

print("CLIP frozen.")

CLIP frozen.


In [15]:
class LungDataset(Dataset):

    def __init__(self, root_dir, processor):

        self.samples = []
        self.processor = processor

        for class_name in CLASSES:

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            if not os.path.exists(class_dir):
                continue

            for filename in os.listdir(class_dir):

                if filename.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp")
                ):

                    image_path = os.path.join(
                        class_dir,
                        filename
                    )

                    label = CLASS_TO_IDX[class_name]

                    self.samples.append(
                        (image_path, label)
                    )

        print(
            f"{os.path.basename(root_dir)}: "
            f"{len(self.samples)} images"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(
            image_path
        ).convert("RGB")

        # CLIP preprocessing
        inputs = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = inputs["pixel_values"].squeeze(0)

        return (
            pixel_values,
            torch.tensor(label, dtype=torch.long)
        )

In [16]:
train_dataset = LungDataset(
    TRAIN_DIR,
    processor
)

val_dataset = LungDataset(
    VAL_DIR,
    processor
)

test_dataset = LungDataset(
    TEST_DIR,
    processor
)

train: 6054 images
val: 2016 images
test: 2025 images


In [17]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created.")

DataLoaders created.


In [18]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels[:10])

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Labels: tensor([0, 2, 1, 3, 4, 4, 3, 0, 1, 1])


In [19]:
images = images.to(DEVICE)

with torch.no_grad():

    outputs = clip_model.get_image_features(
        pixel_values=images
    )

print(type(outputs))
print(outputs)

<class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 3.8124e-01,  7.7047e-02, -1.1181e-03,  ...,  5.7450e-01,
          -3.6719e-02,  4.2309e-01],
         [ 3.5386e-01,  1.9597e-01, -1.4373e-01,  ..., -1.6625e-03,
           4.4577e-01,  3.4311e-01],
         [ 1.3659e-01,  4.5113e-01,  4.6220e-02,  ...,  2.4050e-01,
           7.8180e-02,  5.5344e-01],
         ...,
         [-9.8856e-02,  2.1345e-01, -5.6159e-01,  ...,  2.0937e-01,
           4.0895e-01,  9.0829e-02],
         [ 3.4298e-01,  4.1818e-01, -4.4897e-01,  ...,  2.2988e-01,
           1.4468e-01,  4.0305e-01],
         [ 1.0888e-01,  2.0644e-01, -3.8370e-01,  ...,  5.6505e-01,
           2.6887e-01,  4.9901e-02]],

        [[ 2.8950e-01,  1.4184e-01,  1.3868e-01,  ...,  6.3226e-01,
          -1.6314e-02,  3.6558e-01],
         [ 3.1024e-01,  1.1297e+00, -3.2238e-01,  ...,  8.3546e-01,
           8.1569e-01, -5.1222e-01],
         [ 5.8548e-02,  6.6744e-

In [20]:
features = outputs.pooler_output

print("Feature shape:", features.shape)

Feature shape: torch.Size([32, 512])


In [21]:
with torch.no_grad():

    dummy = torch.zeros(
        1,
        3,
        224,
        224
    ).to(DEVICE)

    dummy_output = clip_model.get_image_features(
        pixel_values=dummy
    )

    feature_dim = dummy_output.pooler_output.shape[-1]

print("CLIP feature dimension:", feature_dim)

CLIP feature dimension: 512


In [22]:
classifier = nn.Sequential(
    
    nn.Linear(
        feature_dim,
        256
    ),

    nn.ReLU(),

    nn.Dropout(0.3),

    nn.Linear(
        256,
        5
    )
)

classifier = classifier.to(DEVICE)

print(classifier)

Sequential(
  (0): Linear(in_features=512, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=256, out_features=5, bias=True)
)


In [23]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Loss and optimizer ready.")

Loss and optimizer ready.


In [24]:
def train_one_epoch():

    classifier.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc="Training"
    )

    for images, labels in progress_bar:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.no_grad():

            clip_outputs = clip_model.get_image_features(
                pixel_values=images
            )

            features = clip_outputs.pooler_output

        # Normalize CLIP embeddings
        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        # -----------------------------
        # Classifier
        # -----------------------------

        logits = classifier(features)

        loss = criterion(
            logits,
            labels
        )

        # -----------------------------
        # Backpropagation
        # -----------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # -----------------------------
        # Metrics
        # -----------------------------

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [25]:
def validate():

    classifier.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        progress_bar = tqdm(
            val_loader,
            desc="Validation"
        )

        for images, labels in progress_bar:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            # CLIP features
            clip_outputs = clip_model.get_image_features(
                pixel_values=images
            )

            features = clip_outputs.pooler_output

            features = features / features.norm(
                dim=-1,
                keepdim=True
            )

            # Classifier
            logits = classifier(features)

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = logits.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    val_loss = running_loss / total
    val_accuracy = correct / total

    return val_loss, val_accuracy

In [ ]:
EPOCHS = 20

best_val_accuracy = 0.0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    train_loss, train_accuracy = train_one_epoch()

    val_loss, val_accuracy = validate()

    # Save history
    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    print(
        f"\nTrain Loss: {train_loss:.4f}"
    )

    print(
        f"Train Accuracy: "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Accuracy: "
        f"{val_accuracy * 100:.2f}%"
    )

    # Save best model
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            classifier.state_dict(),
            "clip_5class_best_classifier.pth"
        )

        print(
            f"✓ Best model saved "
            f"({best_val_accuracy * 100:.2f}%)"
        )

print("\nTraining finished.")
print(
    f"Best validation accuracy: "
    f"{best_val_accuracy * 100:.2f}%"
)


EPOCH 1/20


Validation: 100%|██████████| 63/63 [00:26<00:00,  2.39it/s]



Train Loss: 1.5804
Train Accuracy: 41.54%
Val Loss: 1.5394
Val Accuracy: 57.74%
✓ Best model saved (57.74%)

EPOCH 2/20


Validation: 100%|██████████| 63/63 [00:26<00:00,  2.42it/s]



Train Loss: 1.4721
Train Accuracy: 57.73%
Val Loss: 1.3965
Val Accuracy: 59.13%
✓ Best model saved (59.13%)

EPOCH 3/20


Training:  10%|█         | 19/190 [00:07<01:09,  2.44it/s, loss=1.3199]

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_accuracy"],
    label="Train Accuracy"
)

plt.plot(
    history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CLIP 5-Class Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Train Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CLIP 5-Class Loss")
plt.legend()
plt.show()

In [ ]:
classifier.load_state_dict(
    torch.load(
        "clip_5class_best_classifier.pth",
        map_location=DEVICE
    )
)

classifier.eval()

print("Best classifier loaded.")

In [ ]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=CLASSES,
        digits=4
    )
)